# Rebuild `smap_by_sample.parquet` — shared cells, conflicts, low RAM

This revision addresses the error: **“A single SMAP pixel/date has different values in the CSV exports.”**

The uploaded notebooks map stations to the SMAP grid using `smap_x_idx` and `smap_y_idx`. Normally, stations sharing a cell/date should receive one consistent value. If the station-level GEE exports disagree within a mapped cell/date, the notebook does **not average those values and does not abort the entire rebuild**.

Instead:
- It checks duplicate records for the *same `station_key` + date*. If these disagree, it stops because the same station/date has ambiguous source values.
- For pixel/date groups that are consistent across stations, it shares the single value across all stations mapped to that pixel.
- For pixel/date groups that conflict across stations, it preserves each station's own exported value for that date, writes a diagnostic CSV, and marks affected output rows with `SMAP_cell_conflict = True`. It does not propagate one station's value to another station in a conflicting group.
- It preserves every row of `station_date_candidates.parquet`.

## Memory

DuckDB performs multi-million-row joins/aggregation with a configurable memory limit and disk spilling. The candidate Parquet is not loaded fully into pandas. Ensure there is enough free disk space for temporary work and the output.


In [1]:
from pathlib import Path
from datetime import datetime
import csv
import shutil
import tempfile
import os

try:
    import duckdb
except ImportError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "duckdb"])
    import duckdb

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name != "Global" or not (PROJECT_DIR / "station_date_candidates.parquet").exists():
    candidates = [
        Path.home() / "Code/UG_SM/Data/Global",
        Path.cwd() / "Data/Global",
    ]
    PROJECT_DIR = next((p for p in candidates if (p / "station_date_candidates.parquet").exists()), PROJECT_DIR)

SMAP_DIR = PROJECT_DIR / "smap"
SAMPLES_PATH = PROJECT_DIR / "station_date_candidates.parquet"
MAPPING_PATH = SMAP_DIR / "station_smap_grid_mapping.parquet"
OUTPUT_PATH = SMAP_DIR / "smap_by_sample.parquet"
OUTPUT_TMP = SMAP_DIR / "smap_by_sample.rebuild_tmp.parquet"
CONFLICT_REPORT = SMAP_DIR / "smap_cell_date_conflicts.csv"

print("Project directory:", PROJECT_DIR.resolve())
print("Sample Parquet exists:", SAMPLES_PATH.exists())
print("Station/grid mapping exists:", MAPPING_PATH.exists())
print("Output target:", OUTPUT_PATH.resolve())

if not SAMPLES_PATH.exists():
    raise FileNotFoundError(f"Not found: {SAMPLES_PATH}")
if not MAPPING_PATH.exists():
    raise FileNotFoundError(f"Not found: {MAPPING_PATH}")


Project directory: /home/mayur/Code/UG_SM/Data/Global
Sample Parquet exists: True
Station/grid mapping exists: True
Output target: /home/mayur/Code/UG_SM/Data/Global/smap/smap_by_sample.parquet


In [2]:
# Discover only CSVs containing the expected GEE export columns.
# This avoids treating unrelated report/error CSVs as inputs.
required_csv_columns = {"date", "sm_surface", "smap_n", "station_key"}
csv_files = []

for path in sorted(SMAP_DIR.glob("*.csv")):
    try:
        with path.open("r", newline="", encoding="utf-8-sig") as f:
            headers = set(next(csv.reader(f), []))
        if required_csv_columns.issubset(headers):
            csv_files.append(path)
    except (OSError, UnicodeDecodeError, StopIteration):
        continue

print(f"Matching GEE CSV exports found: {len(csv_files)}")
for path in csv_files:
    print(f"  {path.name} ({path.stat().st_size:,} bytes)")

if len(csv_files) != 6:
    raise ValueError(
        f"Expected the 6 GEE CSV exports with columns {sorted(required_csv_columns)}, "
        f"but found {len(csv_files)}. Check the directory listing above before continuing."
    )


Matching GEE CSV exports found: 6
  SMAP_L4_DAILY_2015_2020_B01.csv (195,840,307 bytes)
  SMAP_L4_DAILY_2015_2020_B02.csv (200,990,692 bytes)
  SMAP_L4_DAILY_2015_2020_B03.csv (41,550,431 bytes)
  SMAP_L4_DAILY_2020_end_B01.csv (255,156,238 bytes)
  SMAP_L4_DAILY_2020_end_B02.csv (261,880,382 bytes)
  SMAP_L4_DAILY_2020_end_B03.csv (54,139,516 bytes)


In [3]:
# Create an isolated DuckDB work area. It is deleted after a successful run.
# Lower this to '512MB' and/or one thread if your machine has very little RAM.
MEMORY_LIMIT = "3GB"
DUCKDB_THREADS = 2

WORK_DIR = Path(tempfile.mkdtemp(prefix=".smap_rebuild_work_", dir=SMAP_DIR))
WORK_DB = WORK_DIR / "smap_rebuild.duckdb"
DUCKDB_TEMP_DIR = WORK_DIR / "spill"
DUCKDB_TEMP_DIR.mkdir(parents=True, exist_ok=True)

def sql_literal(value):
    return "'" + str(value).replace("'", "''") + "'"

con = duckdb.connect(str(WORK_DB))
con.execute(f"SET memory_limit = {sql_literal(MEMORY_LIMIT)}")
con.execute(f"SET temp_directory = {sql_literal(DUCKDB_TEMP_DIR.resolve())}")
con.execute(f"SET threads = {int(DUCKDB_THREADS)}")
con.execute("SET TimeZone = 'UTC'")

print("DuckDB version:", duckdb.__version__)
print("Memory cap:", MEMORY_LIMIT)
print("Threads:", DUCKDB_THREADS)
print("Temporary workspace:", WORK_DIR.resolve())


DuckDB version: 1.5.6
Memory cap: 3GB
Threads: 2
Temporary workspace: /home/mayur/Code/UG_SM/Data/Global/smap/.smap_rebuild_work_zkorszc8


In [4]:
# Build a one-row-per-station mapping table.
# The station_key convention follows the uploaded notebooks:
# station_key = network + "::" + station_id.
con.execute(f'''
CREATE OR REPLACE TABLE station_map AS
SELECT
    TRIM(CAST(network AS VARCHAR)) AS network,
    TRIM(CAST(station_id AS VARCHAR)) AS station_id,
    TRIM(CAST(station_key AS VARCHAR)) AS station_key,
    TRY_CAST(smap_x_idx AS INTEGER) AS pixel_x,
    TRY_CAST(smap_y_idx AS INTEGER) AS pixel_y,
    TRY_CAST(latitude AS DOUBLE) AS latitude,
    TRY_CAST(longitude AS DOUBLE) AS longitude,
    TRY_CAST(smap_distance_m AS DOUBLE) AS station_to_cell_center_m
FROM read_parquet({sql_literal(MAPPING_PATH.resolve())})
''')

mapping_rows = con.execute("SELECT COUNT(*) FROM station_map").fetchone()[0]
mapping_dupe_keys = con.execute('''
    SELECT COUNT(*) FROM (
      SELECT station_key FROM station_map
      GROUP BY station_key HAVING COUNT(*) > 1
    )
''').fetchone()[0]
mapping_dupe_station_ids = con.execute('''
    SELECT COUNT(*) FROM (
      SELECT network, station_id FROM station_map
      GROUP BY network, station_id HAVING COUNT(*) > 1
    )
''').fetchone()[0]
invalid_pixels = con.execute('''
    SELECT COUNT(*) FROM station_map
    WHERE pixel_x IS NULL OR pixel_y IS NULL OR station_key IS NULL OR station_key = ''
''').fetchone()[0]

print(f"Mapping rows: {mapping_rows:,}")
print(f"Duplicated station_key groups: {mapping_dupe_keys:,}")
print(f"Duplicated network/station_id groups: {mapping_dupe_station_ids:,}")
print(f"Stations with missing key/pixel index: {invalid_pixels:,}")

if mapping_dupe_keys or mapping_dupe_station_ids or invalid_pixels:
    raise ValueError("The station/grid mapping is not one-to-one or has missing pixel indices. Inspect the mapping first.")


Mapping rows: 2,210
Duplicated station_key groups: 0
Duplicated network/station_id groups: 0
Stations with missing key/pixel index: 0


In [5]:
# Diagnose co-located stations by exact grid-cell indices.
cell_summary = con.execute('''
WITH cell_counts AS (
    SELECT pixel_x, pixel_y, COUNT(*) AS n_stations
    FROM station_map
    GROUP BY pixel_x, pixel_y
)
SELECT
    (SELECT COUNT(*) FROM station_map) AS stations,
    COUNT(*) AS unique_grid_cells,
    SUM(CASE WHEN n_stations > 1 THEN 1 ELSE 0 END) AS shared_cells,
    SUM(CASE WHEN n_stations > 1 THEN n_stations ELSE 0 END) AS stations_in_shared_cells,
    MAX(n_stations) AS max_stations_per_cell
FROM cell_counts
''').fetchdf()
display(cell_summary)

shared_cells = con.execute('''
SELECT
    pixel_x,
    pixel_y,
    COUNT(*) AS n_stations,
    STRING_AGG(station_key, ' | ' ORDER BY station_key) AS station_keys,
    ROUND(MIN(station_to_cell_center_m), 1) AS min_cell_center_distance_m,
    ROUND(MAX(station_to_cell_center_m), 1) AS max_cell_center_distance_m
FROM station_map
GROUP BY pixel_x, pixel_y
HAVING COUNT(*) > 1
ORDER BY n_stations DESC, pixel_y, pixel_x
LIMIT 100
''').fetchdf()

print("First 100 shared cells (if any):")
display(shared_cells)


,stations,unique_grid_cells,shared_cells,stations_in_shared_cells,max_stations_per_cell
0,2210,1503,178.0,885.0,75


First 100 shared cells (if any):


,pixel_x,pixel_y,n_stations,station_keys,min_cell_center_distance_m,max_cell_center_distance_m
0,2090,206,75,COSMOS::COSMOS__Petzenkirchen | GROW::GROW__0e...,3128.4,5591.1
1,2204,279,39,GROW::GROW__0j1g6k7t | GROW::GROW__0wmxrc70 | ...,4462.7,5053.4
2,2148,208,36,GROW::GROW__1ke3njhh | GROW::GROW__40nmrsbk | ...,290.4,5222.5
3,632,307,30,SOILSCAPE::SOILSCAPE__node1017 | SOILSCAPE::SO...,4671.9,5133.3
4,632,306,26,COSMOS::COSMOS__Tonzi_Ranch | FLUXNET-AMERIFLU...,2700.6,4658.5
...,...,...,...,...,...,...
95,1936,266,2,XMS-CAT::XMS-CAT__Clot_de_les_Peres_I | XMS-CA...,4327.9,4605.5
96,732,269,2,SNOTEL::SNOTEL__Tony_Grove_Lake | SNOTEL::SNOT...,4466.2,5175.9
97,733,269,2,COSMOS::COSMOS__Daniel_Forest | SNOTEL::SNOTEL...,1610.8,1973.0
98,1161,273,2,USCRN::USCRN__Kingston_1_NW | USCRN::USCRN__Ki...,1888.7,2260.9


In [6]:
# Read the six CSVs into a disk-backed DuckDB table.
# filename=true records the source file for auditing possible overlapping exports.
csv_list_sql = "[" + ", ".join(sql_literal(p.resolve()) for p in csv_files) + "]"

con.execute(f'''
CREATE OR REPLACE TABLE smap_raw AS
SELECT
    NULLIF(TRIM(CAST(station_key AS VARCHAR)), '') AS station_key,
    CAST(TRY_CAST("date" AS TIMESTAMPTZ) AS DATE) AS obs_date,
    CASE
        WHEN ISFINITE(TRY_CAST(sm_surface AS DOUBLE))
        THEN TRY_CAST(sm_surface AS DOUBLE)
        ELSE NULL
    END AS sm_surface,
    TRY_CAST(smap_n AS BIGINT) AS smap_n,
    filename AS source_file
FROM read_csv_auto(
    {csv_list_sql},
    union_by_name = true,
    all_varchar = true,
    filename = true
)
''')

raw_stats = con.execute('''
SELECT
    COUNT(*) AS csv_rows,
    SUM(CASE WHEN station_key IS NULL THEN 1 ELSE 0 END) AS missing_station_key_rows,
    SUM(CASE WHEN obs_date IS NULL THEN 1 ELSE 0 END) AS invalid_date_rows,
    SUM(CASE WHEN sm_surface IS NULL THEN 1 ELSE 0 END) AS missing_or_nonfinite_sm_surface_rows,
    SUM(CASE WHEN smap_n IS NULL THEN 1 ELSE 0 END) AS missing_or_noninteger_smap_n_rows
FROM smap_raw
''').fetchdf()
display(raw_stats)

invalid_rows = con.execute('''
SELECT COUNT(*) FROM smap_raw WHERE station_key IS NULL OR obs_date IS NULL
''').fetchone()[0]
if invalid_rows:
    display(con.execute('''
        SELECT * FROM smap_raw
        WHERE station_key IS NULL OR obs_date IS NULL LIMIT 20
    ''').fetchdf())
    raise ValueError(f"{invalid_rows:,} CSV rows have a missing station_key or invalid date. Nothing was written.")

unmapped_csv_rows = con.execute('''
SELECT COUNT(*)
FROM smap_raw r LEFT JOIN station_map m USING (station_key)
WHERE m.station_key IS NULL
''').fetchone()[0]
print(f"CSV rows whose station_key has no grid mapping: {unmapped_csv_rows:,}")
if unmapped_csv_rows:
    display(con.execute('''
        SELECT r.station_key, COUNT(*) AS rows,
               STRING_AGG(DISTINCT r.source_file, ' | ') AS source_files
        FROM smap_raw r LEFT JOIN station_map m USING (station_key)
        WHERE m.station_key IS NULL
        GROUP BY r.station_key ORDER BY rows DESC LIMIT 30
    ''').fetchdf())
    raise ValueError("Some CSV station keys have no grid mapping. No output has been written.")

# Distinguish overlapping/conflicting exports for the same station/date from
# disagreements between different stations that merely map to the same pixel.
con.execute('''
CREATE OR REPLACE TABLE station_day_qc AS
SELECT
    station_key, obs_date,
    COUNT(*) AS source_rows,
    COUNT(DISTINCT sm_surface) AS distinct_sm_surface_values,
    COUNT(DISTINCT smap_n) AS distinct_smap_n_values,
    MIN(sm_surface) AS min_sm_surface,
    MAX(sm_surface) AS max_sm_surface,
    MIN(smap_n) AS min_smap_n,
    MAX(smap_n) AS max_smap_n,
    STRING_AGG(DISTINCT source_file, ' | ') AS source_files
FROM smap_raw
GROUP BY station_key, obs_date
''')

station_day_conflicts = con.execute('''
SELECT COUNT(*) FROM station_day_qc
WHERE distinct_sm_surface_values > 1 OR distinct_smap_n_values > 1
''').fetchone()[0]
print(f"Conflicting duplicate station_key/date groups: {station_day_conflicts:,}")
if station_day_conflicts:
    display(con.execute('''
        SELECT * FROM station_day_qc
        WHERE distinct_sm_surface_values > 1 OR distinct_smap_n_values > 1
        ORDER BY source_rows DESC LIMIT 50
    ''').fetchdf())
    raise ValueError(
        "The same station_key/date has conflicting values across CSV rows. "
        "Inspect the listed source files; the notebook will not choose one arbitrarily."
    )

# Duplicate rows for a station/date are now proven consistent, so collapse them safely.
con.execute('''
CREATE OR REPLACE TABLE smap_station_day AS
SELECT station_key, obs_date,
       MAX(sm_surface) AS sm_surface,
       MAX(smap_n) AS smap_n,
       COUNT(*) AS duplicate_source_rows
FROM smap_raw
GROUP BY station_key, obs_date
''')
print("Unique station/date records:", f"{con.execute('SELECT COUNT(*) FROM smap_station_day').fetchone()[0]:,}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,csv_rows,missing_station_key_rows,invalid_date_rows,missing_or_nonfinite_sm_surface_rows,missing_or_noninteger_smap_n_rows
0,9286636,0.0,0.0,0.0,0.0


CSV rows whose station_key has no grid mapping: 0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Conflicting duplicate station_key/date groups: 0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Unique station/date records: 9,286,636


In [7]:
# Compare station-level values within each mapped pixel/date.
# Different stations may disagree even though the static index mapping assigns them
# the same cell. Do not average or choose an arbitrary station value: mark the group.
con.execute('''
CREATE OR REPLACE TABLE smap_cell_qc AS
SELECT
    m.pixel_x,
    m.pixel_y,
    s.obs_date,
    COUNT(*) AS station_rows_with_export,
    COUNT(DISTINCT s.sm_surface) AS distinct_sm_surface_values,
    COUNT(DISTINCT s.smap_n) AS distinct_smap_n_values,
    MIN(s.sm_surface) AS min_sm_surface,
    MAX(s.sm_surface) AS max_sm_surface,
    MIN(s.smap_n) AS min_smap_n,
    MAX(s.smap_n) AS max_smap_n,
    (COUNT(DISTINCT s.sm_surface) > 1 OR COUNT(DISTINCT s.smap_n) > 1) AS is_conflict
FROM smap_station_day s
JOIN station_map m USING (station_key)
GROUP BY m.pixel_x, m.pixel_y, s.obs_date
''')

cell_day_total = con.execute("SELECT COUNT(*) FROM smap_cell_qc").fetchone()[0]
cell_day_conflicts = con.execute("SELECT COUNT(*) FROM smap_cell_qc WHERE is_conflict").fetchone()[0]
print(f"Pixel/date groups with exported values: {cell_day_total:,}")
print(f"Pixel/date groups with different values across stations: {cell_day_conflicts:,}")

if cell_day_conflicts:
    # Save the full diagnostic table as a CSV; display only its first 100 rows here.
    conflict_sql = f'''
    COPY (
        SELECT
            q.pixel_x, q.pixel_y, q.obs_date,
            q.station_rows_with_export,
            q.distinct_sm_surface_values, q.distinct_smap_n_values,
            m.station_key, m.network, m.station_id,
            m.latitude, m.longitude, m.station_to_cell_center_m,
            s.sm_surface, s.smap_n, s.duplicate_source_rows
        FROM smap_cell_qc q
        JOIN station_map m
          ON m.pixel_x = q.pixel_x AND m.pixel_y = q.pixel_y
        JOIN smap_station_day s
          ON s.station_key = m.station_key AND s.obs_date = q.obs_date
        WHERE q.is_conflict
        ORDER BY q.obs_date, q.pixel_y, q.pixel_x, m.station_key
    ) TO {sql_literal((SMAP_DIR / 'smap_cell_date_conflicts.csv').resolve())}
    (HEADER, DELIMITER ',')
    '''
    con.execute(conflict_sql)
    print("Full conflict diagnostics saved to:", (SMAP_DIR / 'smap_cell_date_conflicts.csv').resolve())
    display(con.execute('''
        SELECT
            q.pixel_x, q.pixel_y, q.obs_date,
            q.distinct_sm_surface_values, q.distinct_smap_n_values,
            m.station_key, m.latitude, m.longitude, m.station_to_cell_center_m,
            s.sm_surface, s.smap_n
        FROM smap_cell_qc q
        JOIN station_map m ON m.pixel_x=q.pixel_x AND m.pixel_y=q.pixel_y
        JOIN smap_station_day s ON s.station_key=m.station_key AND s.obs_date=q.obs_date
        WHERE q.is_conflict
        ORDER BY q.obs_date, q.pixel_y, q.pixel_x, m.station_key
        LIMIT 100
    ''').fetchdf())
else:
    print("No shared-pixel value conflicts detected.")

# Only consistent pixel/date groups enter this reusable lookup table.
# Groups marked is_conflict are intentionally excluded and will use station-level values.
con.execute('''
CREATE OR REPLACE TABLE smap_cells AS
SELECT
    pixel_x,
    pixel_y,
    obs_date,
    max_sm_surface AS SMAP,
    max_smap_n AS SMAP_n_obs
FROM smap_cell_qc
WHERE NOT is_conflict
''')
print("Consistent pixel/date values available to share:", f"{con.execute('SELECT COUNT(*) FROM smap_cells').fetchone()[0]:,}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pixel/date groups with exported values: 6,314,408
Pixel/date groups with different values across stations: 336,233
Full conflict diagnostics saved to: /home/mayur/Code/UG_SM/Data/Global/smap/smap_cell_date_conflicts.csv


,pixel_x,pixel_y,obs_date,distinct_sm_surface_values,distinct_smap_n_values,station_key,latitude,longitude,station_to_cell_center_m,sm_surface,smap_n
0,2026,136,2015-04-01,2,1,HOBE::HOBE__1.02,56.03760,9.16100,5076.577141,0.233383,8
1,2026,136,2015-04-01,2,1,HOBE::HOBE__1.03,56.02830,9.16540,5349.204580,0.233383,8
2,2026,136,2015-04-01,2,1,HOBE::HOBE__1.05,56.03300,9.19120,4151.094381,0.229522,8
3,2026,136,2015-04-01,2,1,HOBE::HOBE__1.06,56.05130,9.16100,4389.141398,0.233383,8
4,2026,136,2015-04-01,2,1,HOBE::HOBE__1.10,56.03480,9.23920,5777.992552,0.229522,8
...,...,...,...,...,...,...,...,...,...,...,...
95,2148,208,2015-04-01,2,1,GROW::GROW__q2gar4w5,47.85622,20.59224,3414.113312,0.351951,8
96,2148,208,2015-04-01,2,1,GROW::GROW__qfqdd9az,47.89088,20.58098,609.736097,0.351951,8
97,2148,208,2015-04-01,2,1,GROW::GROW__sb7kc01j,47.89349,20.57428,1138.832392,0.351951,8
98,2148,208,2015-04-01,2,1,GROW::GROW__sbssctsn,47.89522,20.57396,1160.115630,0.351951,8


Consistent pixel/date values available to share: 5,978,175


In [8]:
# Check the candidate schema and ensure all candidate stations have a grid mapping.
sample_columns = [
    row[0] for row in con.execute(
        f"DESCRIBE SELECT * FROM read_parquet({sql_literal(SAMPLES_PATH.resolve())})"
    ).fetchall()
]
required_sample_columns = {"sample_id", "station_id", "network", "sample_date"}
missing_sample_columns = required_sample_columns - set(sample_columns)
if missing_sample_columns:
    raise ValueError(f"Candidate Parquet is missing columns: {sorted(missing_sample_columns)}")
if "SMAP" in sample_columns or "SMAP_n_obs" in sample_columns:
    raise ValueError("Candidate Parquet already has SMAP/SMAP_n_obs columns; handle those columns before rebuilding.")

unmapped_sample_rows = con.execute(f'''
SELECT COUNT(*)
FROM read_parquet({sql_literal(SAMPLES_PATH.resolve())}) s
LEFT JOIN station_map m
  ON TRIM(CAST(s.network AS VARCHAR)) = m.network
 AND TRIM(CAST(s.station_id AS VARCHAR)) = m.station_id
WHERE m.station_key IS NULL
''').fetchone()[0]

print(f"Candidate columns: {len(sample_columns)}")
print(f"Candidate sample rows without a station/grid mapping: {unmapped_sample_rows:,}")

if unmapped_sample_rows:
    unmapped_sample_stations = con.execute(f'''
        SELECT s.network, s.station_id, COUNT(*) AS sample_rows
        FROM read_parquet({sql_literal(SAMPLES_PATH.resolve())}) s
        LEFT JOIN station_map m
          ON TRIM(CAST(s.network AS VARCHAR)) = m.network
         AND TRIM(CAST(s.station_id AS VARCHAR)) = m.station_id
        WHERE m.station_key IS NULL
        GROUP BY s.network, s.station_id
        ORDER BY sample_rows DESC
        LIMIT 30
    ''').fetchdf()
    display(unmapped_sample_stations)
    raise ValueError("Some candidate stations do not have a pixel mapping. No output has been written.")

source_rows = con.execute(
    f"SELECT COUNT(*) FROM read_parquet({sql_literal(SAMPLES_PATH.resolve())})"
).fetchone()[0]
source_distinct_ids = con.execute(
    f"SELECT COUNT(DISTINCT sample_id) FROM read_parquet({sql_literal(SAMPLES_PATH.resolve())})"
).fetchone()[0]
print(f"Candidate rows: {source_rows:,}")
print(f"Distinct sample_id values: {source_distinct_ids:,}")
if source_rows != source_distinct_ids:
    raise ValueError("sample_id is not unique in station_date_candidates.parquet; review before merging.")


Candidate columns: 15
Candidate sample rows without a station/grid mapping: 0
Candidate rows: 3,514,643
Distinct sample_id values: 3,514,643


In [9]:
# Write directly from SQL so the multi-million-row candidate table is never materialized in pandas.
# For consistent pixel/date groups, use the common pixel value. For conflict groups,
# retain the current station's own exported value and mark the sample row.
if OUTPUT_TMP.exists():
    OUTPUT_TMP.unlink()

station_key_select = "" if "station_key" in sample_columns else ", m.station_key AS station_key"

copy_sql = f'''
COPY (
    SELECT
        s.*{station_key_select},
        CASE WHEN q.is_conflict THEN own.sm_surface ELSE c.SMAP END AS SMAP,
        CASE WHEN q.is_conflict THEN own.smap_n ELSE c.SMAP_n_obs END AS SMAP_n_obs,
        COALESCE(q.is_conflict, FALSE) AS SMAP_cell_conflict
    FROM read_parquet({sql_literal(SAMPLES_PATH.resolve())}) s
    JOIN station_map m
      ON TRIM(CAST(s.network AS VARCHAR)) = m.network
     AND TRIM(CAST(s.station_id AS VARCHAR)) = m.station_id
    LEFT JOIN smap_cell_qc q
      ON q.pixel_x = m.pixel_x
     AND q.pixel_y = m.pixel_y
     AND q.obs_date = CAST(TRY_CAST(s.sample_date AS TIMESTAMPTZ) AS DATE)
    LEFT JOIN smap_cells c
      ON c.pixel_x = m.pixel_x
     AND c.pixel_y = m.pixel_y
     AND c.obs_date = CAST(TRY_CAST(s.sample_date AS TIMESTAMPTZ) AS DATE)
    LEFT JOIN smap_station_day own
      ON own.station_key = m.station_key
     AND own.obs_date = CAST(TRY_CAST(s.sample_date AS TIMESTAMPTZ) AS DATE)
) TO {sql_literal(OUTPUT_TMP.resolve())}
(FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 100000)
'''
con.execute(copy_sql)
print("Temporary output written:", OUTPUT_TMP.resolve())
print("Temporary output size:", f"{OUTPUT_TMP.stat().st_size:,} bytes")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Temporary output written: /home/mayur/Code/UG_SM/Data/Global/smap/smap_by_sample.rebuild_tmp.parquet
Temporary output size: 29,441,475 bytes


In [10]:
# Validate row counts and sample IDs before replacing the target.
tmp_sql = sql_literal(OUTPUT_TMP.resolve())
check = con.execute(f'''
SELECT
    COUNT(*) AS output_rows,
    COUNT(DISTINCT sample_id) AS output_distinct_sample_ids,
    SUM(CASE WHEN SMAP IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_smap,
    SUM(CASE WHEN SMAP IS NULL THEN 1 ELSE 0 END) AS rows_without_smap,
    SUM(CASE WHEN SMAP_cell_conflict THEN 1 ELSE 0 END) AS rows_marked_cell_conflict
FROM read_parquet({tmp_sql})
''').fetchdf()
display(check)

output_rows = int(check.loc[0, "output_rows"])
output_distinct_ids = int(check.loc[0, "output_distinct_sample_ids"])
if output_rows != source_rows:
    raise ValueError(f"Output has {output_rows:,} rows; source has {source_rows:,}. Target not replaced.")
if output_distinct_ids != source_distinct_ids:
    raise ValueError("Distinct sample_id count changed. Target not replaced.")

print("Validation passed: all candidate rows and sample IDs are preserved.")


,output_rows,output_distinct_sample_ids,rows_with_smap,rows_without_smap,rows_marked_cell_conflict
0,3514643,3514643,3510848.0,3795.0,457318.0


Validation passed: all candidate rows and sample IDs are preserved.


In [11]:
# Safe finalization: preserve the existing (possibly damaged) output, then replace it.
backup_path = None
if OUTPUT_PATH.exists():
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_path = OUTPUT_PATH.with_name(f"{OUTPUT_PATH.stem}.backup_{stamp}{OUTPUT_PATH.suffix}")
    shutil.copy2(OUTPUT_PATH, backup_path)
    print("Previous output backed up to:", backup_path.resolve())

OUTPUT_TMP.replace(OUTPUT_PATH)
print("Rebuilt output:", OUTPUT_PATH.resolve())
print("Output rows:", f"{output_rows:,}")
print("Output size:", f"{OUTPUT_PATH.stat().st_size:,} bytes")

con.close()
shutil.rmtree(WORK_DIR, ignore_errors=True)
print("Temporary DuckDB workspace cleaned up.")


Previous output backed up to: /home/mayur/Code/UG_SM/Data/Global/smap/smap_by_sample.backup_20261011_112042.parquet
Rebuilt output: /home/mayur/Code/UG_SM/Data/Global/smap/smap_by_sample.parquet
Output rows: 3,514,643
Output size: 29,441,475 bytes
Temporary DuckDB workspace cleaned up.


## Interpreting the results

- **Shared cells** counts unique `(smap_x_idx, smap_y_idx)` pairs assigned to multiple stations.
- Pixel/date groups with one distinct value are consolidated and the same cell/date value is assigned to all candidate stations in that pixel.
- If different stations mapped to the same cell/date have conflicting values, `smap_cell_date_conflicts.csv` contains the full station/value/distance diagnostics. Their output rows use each station's own exported value and have `SMAP_cell_conflict = True`; no averaging or arbitrary station choice occurs.
- A conflict for the *same* station/date across CSV source rows is a separate issue. The notebook lists those source files and stops rather than choosing a value arbitrarily.
- The output is validated to preserve candidate row counts and distinct `sample_id` values. Missing SMAP can remain when no station/cell observation exists for a date.
- If RAM remains tight, change `MEMORY_LIMIT` to `"512MB"` and `DUCKDB_THREADS` to `1`; make sure sufficient disk space is available for temporary spilling.
